In [1]:
using Pkg
Pkg.add("DecisionTree")
Pkg.add("CSV")
Pkg.add("DataFrames")
Pkg.add("MLBase")
Pkg.add("NearestNeighbors")
Pkg.add("DataStructures")

using DataStructures
using NearestNeighbors
using MLBase
using DecisionTree
using CSV
using DataFrames

   Resolving package versions...
     Project No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Manifest.toml`
   Resolving pac

In [53]:
SHLD = CSV.read("C:\\Users\\tunak\\OneDrive\\Masaüstü\\Sleep_health_and_lifestyle_dataset.csv" , DataFrame)

Row,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
,Int64,String7,Int64,String31,Float64,Int64,Int64,Int64,String15,String7,Int64,Int64,String15
1,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,None
2,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,None
3,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,None
4,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
5,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
6,6,Male,28,Software Engineer,5.9,4,30,8,Obese,140/90,85,3000,Insomnia
7,7,Male,29,Teacher,6.3,6,40,7,Obese,140/90,82,3500,Insomnia
8,8,Male,29,Doctor,7.8,7,75,6,Normal,120/80,70,8000,None
9,9,Male,29,Doctor,7.8,7,75,6,Normal,120/80,70,8000,None


In [54]:
using DataFrames


function split_bp(bp_string)
    parts = split(bp_string, "/")
    return parse(Int, parts[1]), parse(Int, parts[2])
end


transform!(SHLD, "Blood Pressure" => ByRow(x -> split_bp(x)[1]) => :BP_Systolic)
transform!(SHLD, "Blood Pressure" => ByRow(x -> split_bp(x)[2]) => :BP_Diastolic)


select!(SHLD, Not("Blood Pressure"))



using DataFrames


bmi_mapping = Dict(
    "Normal" => 1,
    "Normal Weight" => 1, 
    "Overweight" => 2,
    "Obese" => 3
)


transform!(SHLD, "BMI Category" => ByRow(x -> bmi_mapping[x]) => "BMI Category")


transform!(SHLD, "Gender" => ByRow(x -> x == "Male" ? 0 : 1) => "Gender")
transform!(SHLD, "Sleep Disorder" => ByRow(x -> x == "None" ? 0 : 1) => "Sleep Disorder")

Row,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,BP_Systolic,BP_Diastolic
,Int64,Int64,Int64,String31,Float64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64
1,1,0,27,Software Engineer,6.1,6,42,6,2,77,4200,0,126,83
2,2,0,28,Doctor,6.2,6,60,8,1,75,10000,0,125,80
3,3,0,28,Doctor,6.2,6,60,8,1,75,10000,0,125,80
4,4,0,28,Sales Representative,5.9,4,30,8,3,85,3000,1,140,90
5,5,0,28,Sales Representative,5.9,4,30,8,3,85,3000,1,140,90
6,6,0,28,Software Engineer,5.9,4,30,8,3,85,3000,1,140,90
7,7,0,29,Teacher,6.3,6,40,7,3,82,3500,1,140,90
8,8,0,29,Doctor,7.8,7,75,6,1,70,8000,0,120,80
9,9,0,29,Doctor,7.8,7,75,6,1,70,8000,0,120,80


In [55]:
using MLUtils

train, test = splitobs(SHLD; at = 0.70, shuffle = true)

(ObsView(::DataFrame, ::Vector{Int64})
 262 observations, ObsView(::DataFrame, ::Vector{Int64})
 112 observations)

In [56]:
train_idx = train.indices
test_idx  = test.indices

train = SHLD[train_idx, :]
test  = SHLD[test_idx, :]

Row,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,BP_Systolic,BP_Diastolic
,Int64,Int64,Int64,String31,Float64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64
1,157,0,39,Lawyer,7.2,8,60,5,1,68,8000,0,130,85
2,153,0,39,Lawyer,7.2,8,60,5,1,68,8000,0,130,85
3,272,1,49,Nurse,6.1,6,90,8,2,75,10000,1,140,95
4,358,1,58,Nurse,8.0,9,75,3,2,68,7000,1,140,95
5,50,0,31,Doctor,7.7,7,75,6,1,70,8000,1,120,80
6,193,0,43,Salesperson,6.5,6,45,7,2,72,6000,1,130,85
7,284,1,50,Nurse,6.0,6,90,8,2,75,10000,1,140,95
8,99,1,36,Teacher,7.1,8,60,4,1,68,7000,0,115,75
9,18,0,29,Doctor,6.0,6,30,8,1,70,8000,1,120,80


In [57]:
features_select = train[:, [
    "Gender",    
    "Age", 
    "Sleep Duration",          
    "Quality of Sleep", 
    "Physical Activity Level", 
    "BMI Category", 
    "Heart Rate",  
    "Stress Level", 
]]


features = Array(features_select)


labels = Array(train[!, "Sleep Disorder"])


features = float.(features)


262×8 Matrix{Float64}:
 1.0  38.0  7.4  7.0  60.0  3.0  84.0  5.0
 0.0  37.0  7.8  8.0  70.0  1.0  68.0  4.0
 1.0  44.0  6.6  7.0  45.0  2.0  65.0  4.0
 1.0  49.0  6.2  6.0  90.0  2.0  75.0  8.0
 0.0  43.0  7.8  8.0  90.0  1.0  70.0  5.0
 1.0  50.0  6.0  6.0  90.0  2.0  75.0  8.0
 0.0  40.0  7.9  8.0  90.0  1.0  68.0  5.0
 0.0  44.0  6.5  6.0  45.0  2.0  72.0  7.0
 0.0  36.0  6.6  5.0  35.0  2.0  74.0  7.0
 1.0  50.0  6.0  6.0  90.0  2.0  75.0  8.0
 0.0  29.0  7.8  7.0  75.0  1.0  70.0  6.0
 0.0  39.0  6.5  5.0  40.0  2.0  80.0  7.0
 1.0  31.0  7.9  8.0  75.0  1.0  69.0  4.0
 ⋮                          ⋮          
 0.0  33.0  6.1  6.0  30.0  1.0  72.0  8.0
 0.0  43.0  6.5  6.0  45.0  2.0  72.0  7.0
 0.0  44.0  6.3  6.0  45.0  2.0  72.0  7.0
 0.0  41.0  7.1  7.0  55.0  2.0  72.0  6.0
 1.0  57.0  8.1  9.0  75.0  2.0  68.0  3.0
 0.0  33.0  6.0  6.0  30.0  1.0  72.0  8.0
 0.0  43.0  7.8  8.0  90.0  1.0  70.0  5.0
 1.0  56.0  8.2  9.0  90.0  1.0  65.0  3.0
 1.0  59.0  8.1  9.0  75.0  2.0  6

In [67]:
features_select2 = test[:, [
    "Gender",    
    "Age", 
    "Sleep Duration",          
    "Quality of Sleep", 
    "Physical Activity Level", 
    "BMI Category", 
    "Heart Rate",  
    "Stress Level", 
]]


features_test = Array(features_select2)


labels_test = Array(test[!, "Sleep Disorder"])


features_test = float.(features_test)

112×8 Matrix{Float64}:
 0.0  39.0  7.2  8.0  60.0  1.0  68.0  5.0
 0.0  39.0  7.2  8.0  60.0  1.0  68.0  5.0
 1.0  49.0  6.1  6.0  90.0  2.0  75.0  8.0
 1.0  58.0  8.0  9.0  75.0  2.0  68.0  3.0
 0.0  31.0  7.7  7.0  75.0  1.0  70.0  6.0
 0.0  43.0  6.5  6.0  45.0  2.0  72.0  7.0
 1.0  50.0  6.0  6.0  90.0  2.0  75.0  8.0
 1.0  36.0  7.1  8.0  60.0  1.0  68.0  4.0
 0.0  29.0  6.0  6.0  30.0  1.0  70.0  8.0
 1.0  45.0  6.5  7.0  45.0  2.0  65.0  4.0
 0.0  38.0  7.3  8.0  60.0  1.0  68.0  5.0
 1.0  38.0  7.1  8.0  60.0  1.0  68.0  4.0
 1.0  53.0  8.5  9.0  30.0  1.0  65.0  3.0
 ⋮                          ⋮          
 1.0  33.0  6.2  6.0  50.0  2.0  76.0  6.0
 0.0  35.0  7.5  8.0  60.0  1.0  70.0  5.0
 1.0  59.0  8.2  9.0  75.0  2.0  68.0  3.0
 1.0  29.0  6.5  5.0  40.0  1.0  80.0  7.0
 0.0  29.0  6.1  6.0  30.0  1.0  70.0  8.0
 0.0  32.0  6.0  6.0  30.0  1.0  72.0  8.0
 1.0  53.0  8.5  9.0  30.0  1.0  65.0  3.0
 1.0  50.0  6.0  6.0  90.0  2.0  75.0  8.0
 0.0  41.0  7.6  8.0  90.0  1.0  7

In [66]:
kdtree = KDTree(features_test')

KDTree{StaticArraysCore.SVector{8, Float64}, Euclidean, Float64, StaticArraysCore.SVector{8, Float64}}
  Number of points: 262
  Dimensions: 8
  Metric: Euclidean(0.0)
  Reordered: true

In [68]:
idxs,dists=knn(kdtree, features_test',5)

([[247, 210, 159, 72, 239], [247, 210, 159, 72, 239], [4, 16, 123, 20, 55], [130, 245, 80, 47, 38], [120, 203, 36, 35, 94], [135, 252, 193, 241, 64], [10, 32, 178, 162, 170], [104, 133, 41, 213, 260], [175, 146, 66, 212, 220], [143, 67, 196, 167, 179]  …  [40, 84, 259, 160, 75], [235, 176, 234, 126, 129], [175, 220, 66, 212, 146], [113, 57, 199, 95, 192], [76, 92, 172, 242, 136], [10, 32, 178, 162, 170], [29, 110, 262, 73, 148], [52, 169, 105, 82, 60], [40, 84, 153, 218, 194], [190, 154, 97, 229, 98]], [[0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.10000000000000053, 0.10000000000000053, 0.0, 0.0, 0.09999999999999964], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.10000000000000053, 0.10000000000000053, 0.0, 0.0, 0.0], [2.8301943396169813, 0.09999999999999964, 0.0, 0.0, 0.0], [0.09999999999999964, 0.09999999999999964, 0.0, 0.09999999999999964, 0.09999999999999964]  …  [0.09999999999999964, 0.09999999999999964, 

In [69]:
idxs

112-element Vector{Vector{Int64}}:
 [247, 210, 159, 72, 239]
 [247, 210, 159, 72, 239]
 [4, 16, 123, 20, 55]
 [130, 245, 80, 47, 38]
 [120, 203, 36, 35, 94]
 [135, 252, 193, 241, 64]
 [10, 32, 178, 162, 170]
 [104, 133, 41, 213, 260]
 [175, 146, 66, 212, 220]
 [143, 67, 196, 167, 179]
 [139, 238, 216, 44, 174]
 [98, 74, 186, 166, 91]
 [76, 92, 172, 242, 136]
 ⋮
 [158, 181, 231, 101, 54]
 [65, 133, 48, 52, 108]
 [40, 84, 259, 160, 75]
 [235, 176, 234, 126, 129]
 [175, 220, 66, 212, 146]
 [113, 57, 199, 95, 192]
 [76, 92, 172, 242, 136]
 [10, 32, 178, 162, 170]
 [29, 110, 262, 73, 148]
 [52, 169, 105, 82, 60]
 [40, 84, 153, 218, 194]
 [190, 154, 97, 229, 98]

In [70]:
dists

112-element Vector{Vector{Float64}}:
 [0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0]
 [0.10000000000000053, 0.10000000000000053, 0.0, 0.0, 0.09999999999999964]
 [0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0]
 [0.10000000000000053, 0.10000000000000053, 0.0, 0.0, 0.0]
 [2.8301943396169813, 0.09999999999999964, 0.0, 0.0, 0.0]
 [0.09999999999999964, 0.09999999999999964, 0.0, 0.09999999999999964, 0.09999999999999964]
 [0.20000000000000018, 0.20000000000000018, 0.0, 0.0, 0.0]
 [1.004987562112089, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0]
 ⋮
 [9.000555538409838, 8.870738413458037, 8.870738413458037, 0.0, 8.48528137423857]
 [2.6627053911388696, 2.6627053911388696, 0.0, 2.467792535850613, 2.449489742783178]
 [0.09999999999999964, 0.09999999999999964, 0.09999999999999964, 0.0, 0.0]
 [5.478138369920935, 5.478138369920935, 4.6, 3.1685959035509716, 0.0]
 [2.8284271247461903, 0.09999999999999964, 0.09999999999999964, 0.0999999999

In [71]:
c=labels[hcat(idxs...)]

5×112 Matrix{Int64}:
 0  0  0  1  0  1  1  1  0  1  0  0  0  …  0  0  0  1  0  0  0  1  0  0  0  0
 0  0  1  1  0  1  1  0  0  1  1  0  0     0  0  1  1  0  0  0  1  0  0  1  0
 0  0  1  1  0  1  0  0  0  1  0  0  0     0  0  1  0  0  0  0  0  0  0  1  0
 0  0  1  1  0  1  1  0  0  1  0  0  0     0  0  1  1  0  0  0  1  0  0  1  0
 0  0  1  1  0  1  1  0  0  1  0  0  0     0  0  1  1  0  0  0  1  0  0  1  0

In [72]:
possible_labels=map(i->counter(c[:,i]),1:size(c,2))

112-element Vector{Accumulator{Int64, Int64}}:
 Accumulator(0 => 5)
 Accumulator(0 => 5)
 Accumulator(0 => 1, 1 => 4)
 Accumulator(1 => 5)
 Accumulator(0 => 5)
 Accumulator(1 => 5)
 Accumulator(0 => 1, 1 => 4)
 Accumulator(0 => 4, 1 => 1)
 Accumulator(0 => 5)
 Accumulator(1 => 5)
 Accumulator(0 => 4, 1 => 1)
 Accumulator(0 => 5)
 Accumulator(0 => 5)
 ⋮
 Accumulator(0 => 5)
 Accumulator(0 => 5)
 Accumulator(0 => 1, 1 => 4)
 Accumulator(0 => 1, 1 => 4)
 Accumulator(0 => 5)
 Accumulator(0 => 5)
 Accumulator(0 => 5)
 Accumulator(0 => 1, 1 => 4)
 Accumulator(0 => 5)
 Accumulator(0 => 5)
 Accumulator(0 => 1, 1 => 4)
 Accumulator(0 => 5)

In [73]:
predictions_NN = [argmax(possible_labels[i]) for i in 1:size(c, 2)]

112-element Vector{Int64}:
 0
 0
 1
 1
 0
 1
 1
 0
 0
 1
 0
 0
 0
 ⋮
 0
 0
 1
 1
 0
 0
 0
 1
 0
 0
 1
 0

In [82]:
y_gercek = Int.(labels_test)
y_tahmin = Int.(predictions_NN)


function confusion_matrix_custom(gercek, tahmin)
    tn = sum((gercek .== 0) .& (tahmin .== 0))
    tp = sum((gercek .== 1) .& (tahmin .== 1))
    fp = sum((gercek .== 0) .& (tahmin .== 1))
    fn = sum((gercek .== 1) .& (tahmin .== 0))
    
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    println("--- SONUÇLAR ---")
    println("Doğruluk (Accuracy): ", round(accuracy, digits=3))
    println("TN (Doğru Sağlam): ", tn)
    println("TP (Doğru Hasta) : ", tp)
    println("FP (Yanlış Alarm): ", fp)
    println("FN (Kaçırılan)   : ", fn)
    
    return [tn fp; fn tp]
end

confusion_matrix_custom(y_gercek, y_tahmin)

--- SONUÇLAR ---
Doğruluk (Accuracy): 0.92
TN (Doğru Sağlam): 66
TP (Doğru Hasta) : 37
FP (Yanlış Alarm): 3
FN (Kaçırılan)   : 6


2×2 Matrix{Int64}:
 66   3
  6  37

In [83]:
#   Doğruluk (Accuracy): %92
#   Modeliniz, önüne gelen her 100 vakadan 92'sini doğru biliyor. 
 
#   Confusion Matrix bize modelin nerede hata yaptığını gösterir. 
 
#   Başarılı Olduğu Yerler:
#   66 Kişi (TN): Gerçekten "Sağlam" olan 66 kişiye doğru bir şekilde "Sağlam" demiş. 
#   37 Kişi (TP): Gerçekten "Hasta" olan 37 kişiyi doğru tespit etmiş.
 
#   Hata Yaptığı Yerler (Kritik Kısım):
#   3 Kişi (FP - Yanlış Alarm): Kişiler aslında sağlam, ama model onlara "hasta" demiş.
#   Yorum: Bu kişiler boş yere endişelenir ve gereksiz testlere girer, ama hayati tehlike yaratmaz.
#   6 Kişi (FN - Kaçırılan Vaka): Kişiler aslında hasta, ama model onlara "sağlam" demiş.
#   Yorum: Burası en kritik nokta. Tıbbi modellerde genellikle bu sayının 0'a yakın olması istenir. Çünkü bu 6 hasta, tedavi olması
#   gerekirken evine sağlıklı olduğunu sanarak gönderilmiş oluyor